This my first week project using NVIDIA'S NEMOTRON-3-NANO-OMINI-30B-A3B MODEL...This is a reasoning model just like GPT-5 this is about calculating the entropy of a statistical data..

In [77]:
import math
from binance.client import Client
import numpy as np
import pandas as pd
from scipy.stats import entropy
import requests
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

Basically this piece of code is to get the 1minute candle data from binance you don't need any API key 
for this purpose....

In [78]:
# Binance client
client = Client()

# Fetch candles
klines = client.get_klines(
    symbol='BTCUSDT',
    interval=Client.KLINE_INTERVAL_1MINUTE,
    limit=500
)

In [79]:
# DataFrame
df = pd.DataFrame(klines, columns=[
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'num_trades',
    'taker_buy_base', 'taker_buy_quote', 'ignore'
])
print(df)

         open_time            open            high             low  \
0    1778145660000  81127.61000000  81127.61000000  81065.10000000   
1    1778145720000  81102.23000000  81102.23000000  81050.00000000   
2    1778145780000  81081.11000000  81089.21000000  81081.10000000   
3    1778145840000  81084.34000000  81092.46000000  81081.98000000   
4    1778145900000  81081.99000000  81097.78000000  81057.99000000   
..             ...             ...             ...             ...   
495  1778175360000  79789.80000000  79813.70000000  79789.80000000   
496  1778175420000  79813.00000000  79869.94000000  79812.95000000   
497  1778175480000  79848.75000000  79858.67000000  79820.01000000   
498  1778175540000  79835.95000000  79864.16000000  79835.94000000   
499  1778175600000  79863.38000000  79899.75000000  79863.38000000   

              close       volume     close_time quote_asset_volume  \
0    81102.23000000  46.14526000  1778145719999   3742078.36501580   
1    81081.10000000

In [80]:
# Convert close prices
df['close'] = df['close'].astype(float)

# Compute log returns
df['returns'] = np.log(df['close'] / df['close'].shift(1))

# Remove NaN
returns = df['returns'].dropna()

# Create histogram bins
hist, bins = np.histogram(returns, bins=20)

# Convert to probabilities
probabilities = hist / hist.sum()

# Remove zero probabilities
probabilities = probabilities[probabilities > 0]

# Manual Shannon Entropy
shannon_entropy = -sum(
    p * math.log2(p)
    for p in probabilities
)

h=entropy(probabilities,base=2)

print("Shannon Entropy:", shannon_entropy)
print("Shannon Entropy using scipy:", h)

close = df["close"].astype(float)

# Log returns
returns = np.log(close / close.shift(1)).dropna()

# Variance
variance = np.var(returns)

# Gaussian entropy
G = 0.5 * math.log2(
    2 * math.pi * math.e * variance
)

print("Gaussian Entropy:", G)

Shannon Entropy: 3.213993615370803
Shannon Entropy using scipy: 3.213993615370804
Gaussian Entropy: -8.907411225270264


since we have got the shannon entropy it is time to analyse this using the NVIDIA'S AI MODEL
here we have computed shannon entropy using scipy.stats and by using pure shannon entropy formula

we are using both gaussian entrop and shannon entropy to understand the market structure

In [81]:
load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

Here we will be using API calls to communicate with openrouter
first we wil send a POST requesst to the openrouter API with endpoint 

In [82]:
# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "messages": [
      {
        "role": "system",
        "content": "You are a helpful friend of mine who can answer questions and help with tasks"
      },
        {
          "role": "user",
          "content": f"""I have computed the shannon entropy and gaussian entropy of a statistical data now using this data we can understand the market structure.whether it is uptrend or downtrend. | Property              | What it Means                           |
| --------------------- | --------------------------------------- |
| Structural randomness | Is price movement organized or chaotic? |
| Volatility randomness | How energetic/dispersed is movement?    |
| Entropy          | Captures               |
| ---------------- | ---------------------- |
| Shannon Entropy  | Structural uncertainty |
| Gaussian Entropy | Volatility uncertainty |
\n this is the shannon entropy""" +str(shannon_entropy)+ """and this is the gaussian entropy:"""+str(G)
        }
      ],
    "reasoning": {"enabled": True}
  })
)

In [83]:
response = response.json()
response = response['choices'][0]['message']["reasoning"]
print(response)

We have the numbers: "this is the shannon entropy3.213993615370803and this is the gaussian entropy:-8.907411225270264". So the shannon entropy value is 3.213993615370803 (maybe with a decimal). The gaussian entropy is -8.907411225270264.

We need to interpret: The question: "I have computed the shannon entropy and gaussian entropy of a statistical data now using this data: data we can understand the market structure.whether it is uptrend or downtrend." So they computed shannon entropy (3.213993615370803) and gaussian entropy (-8.907411225270264). They ask: "whether it is uptrend or downtrend."

We need to answer: Based on the values, what does it indicate? Usually, higher shannon entropy indicates more uncertainty, lower gaussian entropy (more negative) indicates lower entropy = more predictable = stronger trend. shannon entropy lower than gaussian entropy means structural randomness less than volatility randomness => price movement more organized than volatile, so uptrend likely. ment